In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Bronze-Ingestion") \
    .config("spark.sql.catalog.demo.type", "rest") \
    .config("spark.sql.catalog.demo.uri", "http://rest:8181") \
    .config("spark.sql.catalog.demo.warehouse", "s3://lakehouse/") \
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.demo.s3.path-style-access", "true") \
    .config("spark.sql.catalog.demo.s3.access-key-id", "adminn") \
    .config("spark.sql.catalog.demo.s3.secret-access-key", "password") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "adminn") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS bronze")
print("✅ Session Spark prête, namespace bronze créé")

✅ Session Spark prête, namespace bronze créé


In [3]:
from pyspark.sql.functions import current_timestamp, input_file_name

# Lecture depuis le volume local monté (au lieu de s3a://)
df_json = spark.read.option("multiLine", "true").json("/home/iceberg/data/raw/inspection-app/")

df_bronze_missions = df_json \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

df_bronze_missions.printSchema()
df_bronze_missions.show(truncate=False)

df_bronze_missions.writeTo("demo.bronze.inspection_missions").createOrReplace()

print("✅ Table bronze.inspection_missions créée")

root
 |-- anomalies: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- anomaly_id: string (nullable = true)
 |    |    |-- criticality: string (nullable = true)
 |    |    |-- element_id: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- inspection_date: string (nullable = true)
 |-- inspector_id: string (nullable = true)
 |-- mission_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_file: string (nullable = false)

+-----------------------------------------------------------------------------------------------------------------------------------------+---------------+------------+----------+-------+--------------------------+------------------------------------------------------+
|anomalies                                                                                                                                |inspection_date|inspect

✅ Table bronze.inspection_missions créée


In [4]:
df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/iceberg/data/raw/sensors/")

df_bronze_sensors = df_csv \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

df_bronze_sensors.printSchema()
df_bronze_sensors.show()

df_bronze_sensors.writeTo("demo.bronze.sensor_readings").createOrReplace()

print("✅ Table bronze.sensor_readings créée")

root
 |-- sensor_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_file: string (nullable = false)

+---------+-------+-------------------+-----------+--------+--------------------+--------------------+
|sensor_id|zone_id|          timestamp|temperature|humidity| ingestion_timestamp|         source_file|
+---------+-------+-------------------+-----------+--------+--------------------+--------------------+
|     S001| ZONE01|2026-08-05 10:00:00|       31.2|      65|2026-08-17 11:32:...|file:///home/iceb...|
|     S001| ZONE01|2026-08-05 10:05:00|       31.5|      64|2026-08-17 11:32:...|file:///home/iceb...|
|     S002| ZONE02|2026-08-05 10:00:00|       28.9|      70|2026-08-17 11:32:...|file:///home/iceb...|
|     S003| ZONE03|2026-08-05 10:00:00|       34.1|      58|2026-08-1

In [5]:
df_geojson_raw = spark.read.option("multiLine", "true").json("/home/iceberg/data/raw/hbim/")

df_bronze_hbim = df_geojson_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

df_bronze_hbim.printSchema()
df_bronze_hbim.show(truncate=False)

df_bronze_hbim.writeTo("demo.bronze.hbim_zones").createOrReplace()

print("✅ Table bronze.hbim_zones créée")

root
 |-- features: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- geometry: struct (nullable = true)
 |    |    |    |-- coordinates: array (nullable = true)
 |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- type: string (nullable = true)
 |    |    |-- properties: struct (nullable = true)
 |    |    |    |-- level: string (nullable = true)
 |    |    |    |-- material: string (nullable = true)
 |    |    |    |-- zone_id: string (nullable = true)
 |    |    |    |-- zone_name: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- type: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_file: string (nullable = false)

+------------------------------------------------------------------------------------------------

In [6]:
spark.sql("SHOW TABLES IN demo.bronze").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|   bronze|         hbim_zones|      false|
|   bronze|inspection_missions|      false|
|   bronze|    sensor_readings|      false|
|   bronze|         test_table|      false|
+---------+-------------------+-----------+



In [7]:
from pyspark.sql.functions import explode, col, to_date, trim, initcap, upper, when

df_bronze_missions = spark.table("demo.bronze.inspection_missions")

df_silver_anomalies = df_bronze_missions \
    .select(
        # Extraire les champs de la mission
        col("mission_id"),
        col("inspection_date"),
        col("inspector_id"),
        col("zone_id"),
        # Déplier la liste des anomalies -> une ligne par anomalie
        explode(col("anomalies")).alias("anomaly")
    ) \
    .select(
        col("mission_id"),
        # Convertir inspection_date en format de date standard
        to_date(col("inspection_date"), "yyyy-MM-dd").alias("inspection_date"),
        col("inspector_id"),
        col("zone_id"),
        col("anomaly.anomaly_id").alias("anomaly_id"),
        col("anomaly.element_id").alias("element_id"),
        # Normaliser le type d'anomalie (espaces retirés, casse cohérente)
        initcap(trim(col("anomaly.type"))).alias("anomaly_type"),
        # Normaliser la criticité (majuscules, cohérent pour les filtres/graphiques)
        upper(trim(col("anomaly.criticality"))).alias("criticality")
    ) \
    .filter(
        # Vérifier les identifiants : aucun ne doit être vide/null
        col("mission_id").isNotNull() &
        col("zone_id").isNotNull() &
        col("anomaly_id").isNotNull() &
        col("element_id").isNotNull()
    )

df_silver_anomalies.show(truncate=False)

df_silver_anomalies.writeTo("demo.silver.inspection_anomalies").createOrReplace()

print("✅ silver.inspection_anomalies —", df_silver_anomalies.count(), "anomalies valides")

+----------+---------------+------------+-------+----------+----------+--------------------+-----------+
|mission_id|inspection_date|inspector_id|zone_id|anomaly_id|element_id|anomaly_type        |criticality|
+----------+---------------+------------+-------+----------+----------+--------------------+-----------+
|M002      |2026-08-10     |EMP002      |ZONE02 |A003      |TAB004    |Infiltration D'eau  |ÉLEVÉE     |
|M002      |2026-08-10     |EMP002      |ZONE02 |A004      |POU008    |Corrosion D'armature|FAIBLE     |
|M002      |2026-08-10     |EMP002      |ZONE02 |A005      |JOI002    |Dégradation De Joint|MOYENNE    |
|M001      |2026-08-05     |EMP001      |ZONE01 |A001      |MUR001    |Fissure             |ÉLEVÉE     |
|M001      |2026-08-05     |EMP001      |ZONE01 |A002      |COL015    |Éclatement Du Béton |MOYENNE    |
+----------+---------------+------------+-------+----------+----------+--------------------+-----------+

✅ silver.inspection_anomalies — 5 anomalies valides


In [8]:
from pyspark.sql.functions import to_timestamp, to_date as to_date_col

df_bronze_sensors = spark.table("demo.bronze.sensor_readings")

df_silver_sensors = df_bronze_sensors \
    .withColumn(
        # Convertir timestamp en format date/heure standard
        "timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
    ) \
    .withColumn("reading_date", to_date_col(col("timestamp"))) \
    .filter(
        # Contrôler les valeurs manquantes
        col("temperature").isNotNull() &
        col("humidity").isNotNull() &
        col("zone_id").isNotNull() &
        col("timestamp").isNotNull()
    ) \
    .filter(
        # Vérifier les unités : température en °C plausible, humidité en % (0-100)
        (col("temperature") > -20) & (col("temperature") < 60) &
        (col("humidity") >= 0) & (col("humidity") <= 100)
    )

df_silver_sensors.show()

df_silver_sensors.writeTo("demo.silver.sensor_readings_clean").createOrReplace()

removed = df_bronze_sensors.count() - df_silver_sensors.count()
print(f"✅ silver.sensor_readings_clean — {removed} ligne(s) rejetée(s) (manquantes/aberrantes)")

+---------+-------+-------------------+-----------+--------+--------------------+--------------------+------------+
|sensor_id|zone_id|          timestamp|temperature|humidity| ingestion_timestamp|         source_file|reading_date|
+---------+-------+-------------------+-----------+--------+--------------------+--------------------+------------+
|     S001| ZONE01|2026-08-05 10:00:00|       31.2|      65|2026-08-17 11:32:...|file:///home/iceb...|  2026-08-05|
|     S001| ZONE01|2026-08-05 10:05:00|       31.5|      64|2026-08-17 11:32:...|file:///home/iceb...|  2026-08-05|
|     S002| ZONE02|2026-08-05 10:00:00|       28.9|      70|2026-08-17 11:32:...|file:///home/iceb...|  2026-08-05|
|     S003| ZONE03|2026-08-05 10:00:00|       34.1|      58|2026-08-17 11:32:...|file:///home/iceb...|  2026-08-05|
+---------+-------+-------------------+-----------+--------+--------------------+--------------------+------------+

✅ silver.sensor_readings_clean — 0 ligne(s) rejetée(s) (manquantes/aber

In [9]:
from pyspark.sql.functions import avg, min as spark_min, max as spark_max, round as spark_round

df_sensor_zone_daily = spark.table("demo.silver.sensor_readings_clean") \
    .groupBy("zone_id", "reading_date") \
    .agg(
        spark_round(avg("temperature"), 1).alias("temp_avg"),
        spark_round(spark_min("temperature"), 1).alias("temp_min"),
        spark_round(spark_max("temperature"), 1).alias("temp_max"),
        spark_round(avg("humidity"), 1).alias("humidity_avg"),
        spark_round(spark_min("humidity"), 1).alias("humidity_min"),
        spark_round(spark_max("humidity"), 1).alias("humidity_max")
    ) \
    .orderBy("zone_id", "reading_date")

df_sensor_zone_daily.show()

df_sensor_zone_daily.writeTo("demo.silver.sensor_zone_daily_stats").createOrReplace()

print("✅ silver.sensor_zone_daily_stats créée — indicateurs par zone/jour")

+-------+------------+--------+--------+--------+------------+------------+------------+
|zone_id|reading_date|temp_avg|temp_min|temp_max|humidity_avg|humidity_min|humidity_max|
+-------+------------+--------+--------+--------+------------+------------+------------+
| ZONE01|  2026-08-05|    31.4|    31.2|    31.5|        64.5|          64|          65|
| ZONE02|  2026-08-05|    28.9|    28.9|    28.9|        70.0|          70|          70|
| ZONE03|  2026-08-05|    34.1|    34.1|    34.1|        58.0|          58|          58|
+-------+------------+--------+--------+--------+------------+------------+------------+

✅ silver.sensor_zone_daily_stats créée — indicateurs par zone/jour


In [10]:
from pyspark.sql.functions import size, expr

df_bronze_hbim = spark.table("demo.bronze.hbim_zones")

df_silver_hbim = df_bronze_hbim \
    .select(explode(col("features")).alias("feature")) \
    .select(
        # Harmoniser zone_id (espaces retirés, majuscules)
        upper(trim(col("feature.properties.zone_id"))).alias("zone_id"),
        col("feature.properties.zone_name").alias("zone_name"),
        col("feature.properties.level").alias("level"),
        col("feature.properties.material").alias("material"),
        col("feature.geometry.type").alias("geometry_type"),
        col("feature.geometry.coordinates").alias("coordinates")
    ) \
    .withColumn(
        # Valider la géométrie : doit être un Polygone avec au moins 4 points
        "geometry_valid",
        (col("geometry_type") == "Polygon") &
        (size(col("coordinates")[0]) >= 4)
    ) \
    .withColumn(
        # Vérifier le système de coordonnées : longitude/latitude dans les bornes valides (WGS84)
        "coordinates_valid",
        expr("""
            forall(coordinates[0], point ->
                point[0] >= -180 AND point[0] <= 180 AND
                point[1] >= -90 AND point[1] <= 90
            )
        """)
    ) \
    .filter(col("zone_id").isNotNull())

df_silver_hbim.show(truncate=False)

df_silver_hbim.writeTo("demo.silver.hbim_zones_clean").createOrReplace()

invalid = df_silver_hbim.filter(~col("geometry_valid") | ~col("coordinates_valid")).count()
print(f"✅ silver.hbim_zones_clean créée — {invalid} zone(s) avec géométrie/coordonnées invalides (à corriger si > 0)")

+-------+-----------+--------+----------+-------------+------------------------------------------------------------------------------------------------------+--------------+-----------------+
|zone_id|zone_name  |level   |material  |geometry_type|coordinates                                                                                           |geometry_valid|coordinates_valid|
+-------+-----------+--------+----------+-------------+------------------------------------------------------------------------------------------------------+--------------+-----------------+
|ZONE01 |Façade Nord|Niveau 1|Béton     |Polygon      |[[[-7.6321, 33.6081], [-7.6323, 33.6081], [-7.6323, 33.6083], [-7.6321, 33.6083], [-7.6321, 33.6081]]]|true          |true             |
|ZONE02 |Façade Sud |Niveau 1|Béton Armé|Polygon      |[[[-7.632, 33.6078], [-7.6322, 33.6078], [-7.6322, 33.608], [-7.632, 33.608], [-7.632, 33.6078]]]     |true          |true             |
+-------+-----------+--------+----------

In [11]:
spark.sql("SHOW TABLES IN demo.silver").show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|   silver|    hbim_zones_clean|      false|
|   silver|inspection_anomalies|      false|
|   silver|sensor_readings_c...|      false|
|   silver|sensor_zone_daily...|      false|
+---------+--------------------+-----------+



In [12]:
df_anomalies = spark.table("demo.silver.inspection_anomalies")
df_hbim = spark.table("demo.silver.hbim_zones_clean")

df_gold_anomalies_enriched = df_anomalies.join(
    df_hbim.select("zone_id", "zone_name", "level", "material"),
    on="zone_id",
    how="left"
)

df_gold_anomalies_enriched.show(truncate=False)

df_gold_anomalies_enriched.writeTo("demo.gold.anomalies_enriched").createOrReplace()

print("✅ gold.anomalies_enriched — anomalies liées à leur zone géographique")

+-------+----------+---------------+------------+----------+----------+--------------------+-----------+-----------+--------+----------+
|zone_id|mission_id|inspection_date|inspector_id|anomaly_id|element_id|anomaly_type        |criticality|zone_name  |level   |material  |
+-------+----------+---------------+------------+----------+----------+--------------------+-----------+-----------+--------+----------+
|ZONE02 |M002      |2026-08-10     |EMP002      |A003      |TAB004    |Infiltration D'eau  |ÉLEVÉE     |Façade Sud |Niveau 1|Béton Armé|
|ZONE02 |M002      |2026-08-10     |EMP002      |A004      |POU008    |Corrosion D'armature|FAIBLE     |Façade Sud |Niveau 1|Béton Armé|
|ZONE02 |M002      |2026-08-10     |EMP002      |A005      |JOI002    |Dégradation De Joint|MOYENNE    |Façade Sud |Niveau 1|Béton Armé|
|ZONE01 |M001      |2026-08-05     |EMP001      |A001      |MUR001    |Fissure             |ÉLEVÉE     |Façade Nord|Niveau 1|Béton     |
|ZONE01 |M001      |2026-08-05     |EMP00

In [13]:
from pyspark.sql.functions import count, countDistinct, lit

df_kpi = spark.sql("""
    SELECT
        COUNT(DISTINCT mission_id)   AS total_missions,
        COUNT(*)                     AS total_anomalies,
        SUM(CASE WHEN criticality IN ('ÉLEVÉE', 'CRITIQUE') THEN 1 ELSE 0 END) AS critical_anomalies,
        COUNT(DISTINCT zone_id)      AS zones_inspected
    FROM demo.silver.inspection_anomalies
""")

df_kpi.show()

df_kpi.writeTo("demo.gold.kpi_summary").createOrReplace()

print("✅ gold.kpi_summary créée")

+--------------+---------------+------------------+---------------+
|total_missions|total_anomalies|critical_anomalies|zones_inspected|
+--------------+---------------+------------------+---------------+
|             2|              5|                 2|              2|
+--------------+---------------+------------------+---------------+

✅ gold.kpi_summary créée


In [14]:
df_gold_by_type = spark.table("demo.silver.inspection_anomalies") \
    .groupBy("anomaly_type") \
    .agg(count("*").alias("nb_anomalies")) \
    .orderBy(col("nb_anomalies").desc())

df_gold_by_type.show()
df_gold_by_type.writeTo("demo.gold.anomalies_by_type").createOrReplace()

df_gold_by_criticality = spark.table("demo.silver.inspection_anomalies") \
    .groupBy("criticality") \
    .agg(count("*").alias("nb_anomalies")) \
    .orderBy(col("nb_anomalies").desc())

df_gold_by_criticality.show()
df_gold_by_criticality.writeTo("demo.gold.anomalies_by_criticality").createOrReplace()

print("✅ gold.anomalies_by_type et gold.anomalies_by_criticality créées")

+--------------------+------------+
|        anomaly_type|nb_anomalies|
+--------------------+------------+
| Éclatement Du Béton|           1|
|Corrosion D'armature|           1|
|  Infiltration D'eau|           1|
|             Fissure|           1|
|Dégradation De Joint|           1|
+--------------------+------------+

+-----------+------------+
|criticality|nb_anomalies|
+-----------+------------+
|    MOYENNE|           2|
|     ÉLEVÉE|           2|
|     FAIBLE|           1|
+-----------+------------+

✅ gold.anomalies_by_type et gold.anomalies_by_criticality créées


In [15]:
# Nombre d'anomalies par zone
df_anomalies_by_zone = spark.table("demo.silver.inspection_anomalies") \
    .groupBy("zone_id") \
    .agg(
        count("*").alias("nb_anomalies"),
        spark_round(
            avg(when(col("criticality").isin("ÉLEVÉE", "CRITIQUE"), 1).otherwise(0)) * 100, 1
        ).alias("pct_anomalies_critiques")
    )

# Moyenne température/humidité par zone (toutes périodes confondues)
df_env_by_zone = spark.table("demo.silver.sensor_zone_daily_stats") \
    .groupBy("zone_id") \
    .agg(
        spark_round(avg("temp_avg"), 1).alias("temp_moyenne"),
        spark_round(avg("humidity_avg"), 1).alias("humidite_moyenne")
    )

# Croisement final : zone géographique + anomalies + environnement
df_gold_zone_summary = df_hbim.select("zone_id", "zone_name", "level", "material") \
    .join(df_anomalies_by_zone, on="zone_id", how="left") \
    .join(df_env_by_zone, on="zone_id", how="left") \
    .fillna(0, subset=["nb_anomalies", "pct_anomalies_critiques"])

df_gold_zone_summary.show(truncate=False)

df_gold_zone_summary.writeTo("demo.gold.zone_summary").createOrReplace()

print("✅ gold.zone_summary — indicateur combiné complet par zone")

+-------+-----------+--------+----------+------------+-----------------------+------------+----------------+
|zone_id|zone_name  |level   |material  |nb_anomalies|pct_anomalies_critiques|temp_moyenne|humidite_moyenne|
+-------+-----------+--------+----------+------------+-----------------------+------------+----------------+
|ZONE01 |Façade Nord|Niveau 1|Béton     |2           |50.0                   |31.4        |64.5            |
|ZONE02 |Façade Sud |Niveau 1|Béton Armé|3           |33.3                   |28.9        |70.0            |
+-------+-----------+--------+----------+------------+-----------------------+------------+----------------+

✅ gold.zone_summary — indicateur combiné complet par zone


In [16]:
spark.sql("SHOW TABLES IN demo.gold").show()

print("\n--- KPI globaux ---")
spark.table("demo.gold.kpi_summary").show()

print("\n--- Synthèse par zone (table clé du dashboard) ---")
spark.table("demo.gold.zone_summary").show(truncate=False)

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|     gold|anomalies_by_crit...|      false|
|     gold|   anomalies_by_type|      false|
|     gold|  anomalies_enriched|      false|
|     gold|         kpi_summary|      false|
|     gold|        zone_summary|      false|
+---------+--------------------+-----------+


--- KPI globaux ---
+--------------+---------------+------------------+---------------+
|total_missions|total_anomalies|critical_anomalies|zones_inspected|
+--------------+---------------+------------------+---------------+
|             2|              5|                 2|              2|
+--------------+---------------+------------------+---------------+


--- Synthèse par zone (table clé du dashboard) ---
+-------+-----------+--------+----------+------------+-----------------------+------------+----------------+
|zone_id|zone_name  |level   |material  |nb_anomalies|pct_anomalies_cri

In [17]:
!pip install Pillow imagehash boto3 --quiet
print("✅ Dépendances installées")


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
✅ Dépendances installées


In [18]:
import boto3
import os
import hashlib
from datetime import datetime
from PIL import Image
from pyspark.sql import Row

# Client S3 pour transférer les fichiers binaires (Spark ne gère pas bien le binaire, boto3 est plus adapté)
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="adminn",
    aws_secret_access_key="password"
)

SOURCE_DIR = "/home/iceberg/data/raw/images"

# 1) Upload des fichiers bruts vers MinIO (zone Bronze)
for filename in os.listdir(SOURCE_DIR):
    filepath = os.path.join(SOURCE_DIR, filename)
    s3_client.upload_file(filepath, "lakehouse", f"bronze/images_raw/{filename}")

print("✅ Images brutes uploadées dans lakehouse/bronze/images_raw/")

# 2) Créer le registre de métadonnées (table Iceberg Bronze)
bronze_metadata = []

for filename in os.listdir(SOURCE_DIR):
    filepath = os.path.join(SOURCE_DIR, filename)
    try:
        with Image.open(filepath) as img:
            img.verify()
        with Image.open(filepath) as img:  # réouverture nécessaire après verify()
            width, height = img.size
            format_ = img.format

        with open(filepath, "rb") as f:
            content_hash = hashlib.sha256(f.read()).hexdigest()

        bronze_metadata.append(Row(
            filename=filename,
            s3_path=f"s3://lakehouse/bronze/images_raw/{filename}",
            width=width, height=height, format=format_,
            content_hash=content_hash, is_valid=True,
            ingestion_timestamp=datetime.now()
        ))
    except Exception:
        bronze_metadata.append(Row(
            filename=filename,
            s3_path=f"s3://lakehouse/bronze/images_raw/{filename}",
            width=None, height=None, format=None,
            content_hash=None, is_valid=False,
            ingestion_timestamp=datetime.now()
        ))

df_bronze_images = spark.createDataFrame(bronze_metadata)
df_bronze_images.show(truncate=False)

spark.sql("CREATE NAMESPACE IF NOT EXISTS bronze")
df_bronze_images.writeTo("demo.bronze.images_metadata").createOrReplace()

print("✅ Table bronze.images_metadata créée")

✅ Images brutes uploadées dans lakehouse/bronze/images_raw/


+----------+-------------------------------------------+-----+------+------+----------------------------------------------------------------+--------+--------------------------+
|filename  |s3_path                                    |width|height|format|content_hash                                                    |is_valid|ingestion_timestamp       |
+----------+-------------------------------------------+-----+------+------+----------------------------------------------------------------+--------+--------------------------+
|IMG001.jpg|s3://lakehouse/bronze/images_raw/IMG001.jpg|227  |227   |JPEG  |e89b071a7eeda20f404c3a7f300d11492e1ff1e598c8cba43d04367b6b344a44|true    |2026-08-17 11:33:40.788628|
|IMG002.jpg|s3://lakehouse/bronze/images_raw/IMG002.jpg|2667 |2000  |JPEG  |d459544e5dea1cbeb4e0e283e5ce880ec51635bdba7d912034ab1978fc56f57d|true    |2026-08-17 11:33:40.803799|
|IMG003.jpg|s3://lakehouse/bronze/images_raw/IMG003.jpg|547  |365   |JPEG  |47fb195d0f5251aeaae71106ddbb6f5c24

In [19]:
import imagehash

TARGET_SIZE = (224, 224)
MIN_RESOLUTION = (64, 64)

silver_metadata = []
valid_rows = df_bronze_images.filter(df_bronze_images.is_valid == True).collect()

for row in valid_rows:
    filepath = os.path.join(SOURCE_DIR, row.filename)

    if row.width < MIN_RESOLUTION[0] or row.height < MIN_RESOLUTION[1]:
        continue  # rejette les images trop petites

    with Image.open(filepath) as img:
        phash = str(imagehash.phash(img))  # empreinte visuelle pour détecter les doublons
        img_resized = img.convert("RGB").resize(TARGET_SIZE)

        local_tmp = f"/tmp/{row.filename}.jpg"
        img_resized.save(local_tmp, "JPEG", quality=95)

    s3_key = f"silver/images_standardized/{row.filename}.jpg"
    s3_client.upload_file(local_tmp, "lakehouse", s3_key)
    os.remove(local_tmp)

    silver_metadata.append(Row(
        filename=f"{row.filename}.jpg",
        s3_path=f"s3://lakehouse/{s3_key}",
        width=TARGET_SIZE[0], height=TARGET_SIZE[1],
        perceptual_hash=phash,
        source_filename=row.filename
    ))

df_silver_images = spark.createDataFrame(silver_metadata)

# Supprime les doublons visuels (même perceptual_hash)
df_silver_images_dedup = df_silver_images.dropDuplicates(["perceptual_hash"])

df_silver_images_dedup.show(truncate=False)

spark.sql("CREATE NAMESPACE IF NOT EXISTS silver")
df_silver_images_dedup.writeTo("demo.silver.images_standardized").createOrReplace()

removed = df_silver_images.count() - df_silver_images_dedup.count()
print(f"✅ silver.images_standardized créée — {removed} doublon(s) retiré(s)")

+--------------+--------------------------------------------------------+-----+------+----------------+---------------+
|filename      |s3_path                                                 |width|height|perceptual_hash |source_filename|
+--------------+--------------------------------------------------------+-----+------+----------------+---------------+
|IMG001.jpg.jpg|s3://lakehouse/silver/images_standardized/IMG001.jpg.jpg|224  |224   |b8b0e56f2a90c46f|IMG001.jpg     |
|IMG003.jpg.jpg|s3://lakehouse/silver/images_standardized/IMG003.jpg.jpg|224  |224   |e996ce69129dc465|IMG003.jpg     |
|IMG002.jpg.jpg|s3://lakehouse/silver/images_standardized/IMG002.jpg.jpg|224  |224   |ead891b765907b44|IMG002.jpg     |
+--------------+--------------------------------------------------------+-----+------+----------------+---------------+

✅ silver.images_standardized créée — 0 doublon(s) retiré(s)


In [20]:
import random

random.seed(42)  # garantit que le split est toujours le même si tu relances
rows = df_silver_images_dedup.collect()
random.shuffle(rows)

n = len(rows)
splits = (
    [("train", r) for r in rows[:int(n*0.7)]] +
    [("val", r) for r in rows[int(n*0.7):int(n*0.85)]] +
    [("test", r) for r in rows[int(n*0.85):]]
)

dataset_version = datetime.now().strftime("v%Y%m%d_%H%M%S")
gold_metadata = []

for split_name, row in splits:
    src_key = row.s3_path.replace("s3://lakehouse/", "")
    dst_key = f"gold/dataset/{split_name}/{row.filename}"

    # Copie directement sur MinIO, sans re-télécharger le fichier (rapide)
    s3_client.copy_object(
        Bucket="lakehouse",
        CopySource={"Bucket": "lakehouse", "Key": src_key},
        Key=dst_key
    )

    gold_metadata.append(Row(
        filename=row.filename,
        s3_path=f"s3://lakehouse/{dst_key}",
        split=split_name,
        perceptual_hash=row.perceptual_hash,
        dataset_version=dataset_version
    ))

df_gold_images = spark.createDataFrame(gold_metadata)
df_gold_images.show(truncate=False)

spark.sql("CREATE NAMESPACE IF NOT EXISTS gold")
df_gold_images.writeTo("demo.gold.images_dataset_manifest").createOrReplace()

print(f"✅ gold.images_dataset_manifest créée — dataset {dataset_version}")
df_gold_images.groupBy("split").count().show()

+--------------+------------------------------------------------+-----+----------------+----------------+
|filename      |s3_path                                         |split|perceptual_hash |dataset_version |
+--------------+------------------------------------------------+-----+----------------+----------------+
|IMG003.jpg.jpg|s3://lakehouse/gold/dataset/train/IMG003.jpg.jpg|train|e996ce69129dc465|v20260817_113411|
|IMG001.jpg.jpg|s3://lakehouse/gold/dataset/train/IMG001.jpg.jpg|train|b8b0e56f2a90c46f|v20260817_113411|
|IMG002.jpg.jpg|s3://lakehouse/gold/dataset/test/IMG002.jpg.jpg |test |ead891b765907b44|v20260817_113411|
+--------------+------------------------------------------------+-----+----------------+----------------+

✅ gold.images_dataset_manifest créée — dataset v20260817_113411
+-----+-----+
|split|count|
+-----+-----+
|train|    2|
| test|    1|
+-----+-----+



In [21]:
spark.sql("SHOW TABLES IN demo.bronze").show()
spark.sql("SHOW TABLES IN demo.silver").show()
spark.sql("SHOW TABLES IN demo.gold").show()

print("\n--- Registre Gold (dataset final) ---")
spark.table("demo.gold.images_dataset_manifest").show(truncate=False)

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|   bronze|         hbim_zones|      false|
|   bronze|    images_metadata|      false|
|   bronze|inspection_missions|      false|
|   bronze|    sensor_readings|      false|
|   bronze|         test_table|      false|
+---------+-------------------+-----------+

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|   silver|    hbim_zones_clean|      false|
|   silver| images_standardized|      false|
|   silver|inspection_anomalies|      false|
|   silver|sensor_readings_c...|      false|
|   silver|sensor_zone_daily...|      false|
+---------+--------------------+-----------+

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|     gold|anomalies_by_crit...|      false|
|     gold|   ano